In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
Generic RF Network Simulation
This script simulates a 3D wireless environment with several types of transmitters:
1. Static stations
2. Low mobility users, for example pedestrians
3. Medium mobility users, for example vehicles
4. High mobility users, for example aircraft

For every scenario, the simulator creates:
- A continuous received IQ signal at one receiver located at (0, 0, 0)
- A sequence of spectrogram images
- A visual 2D scene map of the environment
- RSSI values over time
- One metadata.csv file that includes all labels and simulation parameters

The goal is to create a dataset that can later be used to test whether an AI model
can estimate the type and number of objects from the spectrogram sequence.
"""

import math
import csv
import json
import shutil
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft, get_window

# ============================================================
#                 Global simulation parameters
# ============================================================

# Speed of light [m/s]
c = 3.0e8

# Sampling frequency [Hz]
Fs = 2_000_000

# Total scenario duration [sec]
SCENARIO_DURATION_SEC = 2.0

# Each scenario is divided into several frames.
# Every frame will become one spectrogram image.
FRAME_DURATION_SEC = 0.25
FRAME_STEP_SEC = 0.25

TOTAL_SAMPLES = int(Fs * SCENARIO_DURATION_SEC)
FRAME_SAMPLES = int(Fs * FRAME_DURATION_SEC)
FRAME_STEP_SAMPLES = int(Fs * FRAME_STEP_SEC)

NUM_FRAMES = int(np.floor((TOTAL_SAMPLES - FRAME_SAMPLES) / FRAME_STEP_SAMPLES)) + 1

# STFT parameters
Nwin = 1024
hop = 512
Nfft = 2048
window_name = "hann"

# This is only used as a fallback value.
# The actual SPS is selected according to the object type.
DEFAULT_SPS = 8

# Samples per symbol for each object type.
# Larger SPS means lower symbol rate and narrower bandwidth.
# Smaller SPS means higher symbol rate and wider bandwidth.
SPS_BY_KIND = {
    "static": 16,
    "low": 12,
    "medium": 8,
    "high": 4,
}

# Transmit power ranges in dBm for each object type.
# These values are used for the simulation and not as an exact standard.
TX_POWER_DBM_RANGE_BY_KIND = {
    "static": (30.0, 43.0),    # Static station, usually stronger transmitter
    "low": (0.0, 23.0),        # Pedestrian / mobile device, weaker transmitter
    "medium": (20.0, 33.0),    # Vehicle, medium transmit power
    "high": (35.0, 50.0),      # Aircraft, stronger transmitter
}

# Baseband frequency offsets.
# These offsets create separated frequency regions in the spectrogram.
F_OFFSET_MIN_HZ = -700e3
F_OFFSET_MAX_HZ = 700e3
CHANNEL_SPACING_HZ = 100e3
CHANNEL_JITTER_HZ = 15e3

# Statistical channel effects: shadowing and multipath fading.
ENABLE_FADING = True
SHADOW_STD_DB = 3.0
SHADOW_UPDATE_SEC = 0.05

FADING_UPDATE_SEC = 0.005
DEFAULT_K_DB = 6.0

# ============================================================
#                    Output folders
# ============================================================

# Main output folder in Google Drive.
OUT_BASE_DIR = Path("/content/drive/MyDrive/my_project/data")

OUT_SPEC_DIR = OUT_BASE_DIR / "spectrogram_sequences"
OUT_RSSI_DIR = OUT_BASE_DIR / "rssi"
OUT_SCENE_DIR = OUT_BASE_DIR / "scene_maps"
OUT_META_CSV = OUT_BASE_DIR / "metadata.csv"

OUT_SPEC_DIR.mkdir(parents=True, exist_ok=True)
OUT_RSSI_DIR.mkdir(parents=True, exist_ok=True)
OUT_SCENE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
#                    Transmitter object
# ============================================================

@dataclass
class Emitter3D:
    """
    This class holds all the parameters of one transmitter in the 3D space.
    """
    emitter_id: int
    kind: str
    pos0: np.ndarray
    vel: np.ndarray
    fc: float

    tx_power_dbm: float
    tx_power_watt: float
    power_lin: float

    f_offset_hz: float
    sps: int
    waveform: str = "QPSK"

# ============================================================
#                    Power conversion helpers
# ============================================================

def tx_power_dbm_to_watt(tx_power_dbm: float) -> float:
    """
    Convert transmit power from dBm to Watt.
    0 dBm = 1 mW = 0.001 W.
    """
    return 10.0 ** ((tx_power_dbm - 30.0) / 10.0)

def tx_power_watt_to_amplitude_scale(tx_power_watt: float) -> float:
    """
    In the simulation, the signal is multiplied by an amplitude factor.
    Since power is proportional to |x|^2, the amplitude scale is sqrt(power).
    """
    return float(np.sqrt(tx_power_watt))

def draw_tx_power_dbm(kind: str) -> float:
    """
    Draw a random transmit power according to the object type.
    """
    p_min, p_max = TX_POWER_DBM_RANGE_BY_KIND[kind]
    return float(np.random.uniform(p_min, p_max))

# ============================================================
#                   Physical channel functions
# ============================================================

def fspl_db(fc_hz: float, d_m: np.ndarray) -> np.ndarray:
    """
    Calculate Free Space Path Loss in dB.
    The received signal becomes weaker as the distance gets larger.
    """
    d_m = np.maximum(d_m, 1e-6)
    return 20.0 * np.log10(4.0 * np.pi * fc_hz * d_m / c)

def amp_from_fspl(fc_hz: float, d_m: np.ndarray) -> np.ndarray:
    """
    Convert FSPL from dB to a linear amplitude coefficient.
    """
    fspl = fspl_db(fc_hz, d_m)
    return 10.0 ** (-fspl / 20.0)

def radial_velocity_over_time_xyz(
    x_t: np.ndarray,
    y_t: np.ndarray,
    z_t: np.ndarray,
    vel: np.ndarray,
    rx_pos: np.ndarray = np.array([0.0, 0.0, 0.0]),
) -> np.ndarray:
    """
    Calculate the radial velocity over time.
    Radial velocity is the part of the velocity that points toward or away
    from the receiver.
    """
    vx, vy, vz = vel

    vec_x = rx_pos[0] - x_t
    vec_y = rx_pos[1] - y_t
    vec_z = rx_pos[2] - z_t

    d = np.sqrt(vec_x**2 + vec_y**2 + vec_z**2) + 1e-12

    ux = vec_x / d
    uy = vec_y / d
    uz = vec_z / d

    return vx * ux + vy * uy + vz * uz

def radial_velocity_single(
    tx_pos: np.ndarray,
    tx_vel: np.ndarray,
    rx_pos: np.ndarray = np.array([0.0, 0.0, 0.0]),
) -> float:
    """
    Calculate radial velocity at one specific point in time.
    This is mainly used for saving metadata.
    """
    vec = rx_pos - tx_pos
    d = np.linalg.norm(vec) + 1e-12
    u = vec / d
    return float(np.dot(tx_vel, u))

def doppler_hz(fc_hz: float, v_rad) -> np.ndarray:
    """
    Calculate Doppler frequency shift in Hz.
    """
    return (v_rad / c) * fc_hz

def awgn(x: np.ndarray, snr_db: float | None) -> np.ndarray:
    """
    Add complex AWGN noise according to the required SNR.
    """
    if snr_db is None:
        return x

    xrms = np.sqrt(np.mean(np.abs(x) ** 2) + 1e-18)
    nstd = xrms * 10.0 ** (-snr_db / 20.0) / np.sqrt(2.0)

    n = (np.random.randn(*x.shape) + 1j * np.random.randn(*x.shape)) * nstd
    return x + n

# ============================================================
#                Statistical fading and multipath
# ============================================================

def db_to_lin_power(x_db: float) -> float:
    """
    Convert dB to linear power scale.
    """
    return 10.0 ** (x_db / 10.0)

def db_to_lin_amplitude(x_db):
    """
    Convert dB to linear amplitude scale.
    """
    return 10.0 ** (np.array(x_db) / 20.0)

def generate_slow_shadowing(
    num_samples: int,
    Fs: float,
    std_db: float = SHADOW_STD_DB,
    update_sec: float = SHADOW_UPDATE_SEC,
) -> np.ndarray:
    """
    Generate slow shadowing over time.
    This models slow power changes caused by blocking or large objects.
    """
    step = max(1, int(update_sec * Fs))
    num_knots = int(np.ceil(num_samples / step)) + 1

    shadow_db_knots = np.random.randn(num_knots) * std_db

    x_knots = np.arange(num_knots) * step
    x_full = np.arange(num_samples)

    shadow_db = np.interp(x_full, x_knots, shadow_db_knots)
    shadow_amp = db_to_lin_amplitude(shadow_db)

    return shadow_amp.astype(np.float32)

def generate_rician_fading(
    num_samples: int,
    Fs: float,
    update_sec: float = FADING_UPDATE_SEC,
    k_db: float = DEFAULT_K_DB,
) -> np.ndarray:
    """
    Generate Rician fading.
    This is a simple statistical model for a direct path plus reflections.
    """
    step = max(1, int(update_sec * Fs))
    num_knots = int(np.ceil(num_samples / step)) + 1

    g_re = np.random.randn(num_knots)
    g_im = np.random.randn(num_knots)

    x_knots = np.arange(num_knots) * step
    x_full = np.arange(num_samples)

    g_re_full = np.interp(x_full, x_knots, g_re)
    g_im_full = np.interp(x_full, x_knots, g_im)

    g = (g_re_full + 1j * g_im_full) / np.sqrt(2.0)
    g = g / np.sqrt(np.mean(np.abs(g) ** 2) + 1e-18)

    k_lin = db_to_lin_power(k_db)

    h = np.sqrt(k_lin / (k_lin + 1.0)) + np.sqrt(1.0 / (k_lin + 1.0)) * g
    h = h / np.sqrt(np.mean(np.abs(h) ** 2) + 1e-18)

    return h.astype(np.complex64)

def generate_channel_fading(
    num_samples: int,
    Fs: float,
    enable_fading: bool = ENABLE_FADING,
):
    """
    Generate the channel fading terms.
    Returns:
    - shadow_amp: slow amplitude changes
    - h: complex fading coefficient
    """
    if not enable_fading:
        shadow_amp = np.ones(num_samples, dtype=np.float32)
        h = np.ones(num_samples, dtype=np.complex64)
        return shadow_amp, h

    shadow_amp = generate_slow_shadowing(
        num_samples=num_samples,
        Fs=Fs,
        std_db=SHADOW_STD_DB,
        update_sec=SHADOW_UPDATE_SEC,
    )

    h = generate_rician_fading(
        num_samples=num_samples,
        Fs=Fs,
        update_sec=FADING_UPDATE_SEC,
        k_db=DEFAULT_K_DB,
    )

    return shadow_amp, h

# ============================================================
#                    QPSK baseband signal
# ============================================================

def generate_qpsk_baseband(num_samples: int, sps: int = DEFAULT_SPS) -> np.ndarray:
    """
    Generate a simple QPSK baseband signal.
    Smaller SPS gives a wider signal in frequency.
    """
    num_symbols = math.ceil(num_samples / sps)

    bits = np.random.randint(0, 2, size=(num_symbols, 2))

    mapping = {
        (0, 0):  1 + 1j,
        (0, 1): -1 + 1j,
        (1, 1): -1 - 1j,
        (1, 0):  1 - 1j,
    }

    symbols = np.array([mapping[tuple(b)] for b in bits], dtype=np.complex64)
    symbols /= np.sqrt(2.0)

    x = np.repeat(symbols, sps)

    return x[:num_samples].astype(np.complex64)

# ============================================================
#                    Baseband frequency offset
# ============================================================

def draw_baseband_frequency_offset() -> float:
    """
    Draw a random baseband frequency offset for one transmitter.
    This helps create several visible bands in the spectrogram.
    """
    channels = np.arange(
        F_OFFSET_MIN_HZ,
        F_OFFSET_MAX_HZ + CHANNEL_SPACING_HZ,
        CHANNEL_SPACING_HZ,
    )

    center = float(np.random.choice(channels))
    jitter = float(np.random.uniform(-CHANNEL_JITTER_HZ, CHANNEL_JITTER_HZ))

    return center + jitter

# ============================================================
#                    Emitter generation
# ============================================================

def random_emitter(kind: str, emitter_id: int) -> Emitter3D:
    """
    Create one random transmitter according to its object type.
    """
    if kind == "static":
        speed_min, speed_max = 0.0, 0.0
        r_min, r_max = 50.0, 300.0
        z_min, z_max = 0.0, 10.0
        fc = 900e6
        sps = SPS_BY_KIND["static"]

    elif kind == "low":
        speed_min, speed_max = 0.0, 2.0
        r_min, r_max = 20.0, 500.0
        z_min, z_max = 0.0, 2.0
        fc = 1.8e9
        sps = SPS_BY_KIND["low"]

    elif kind == "medium":
        speed_min, speed_max = 5.0, 30.0
        r_min, r_max = 50.0, 1200.0
        z_min, z_max = 0.0, 5.0
        fc = 2.4e9
        sps = SPS_BY_KIND["medium"]

    elif kind == "high":
        speed_min, speed_max = 70.0, 250.0
        r_min, r_max = 500.0, 3000.0
        z_min, z_max = 100.0, 1000.0
        fc = 5.0e9
        sps = SPS_BY_KIND["high"]

    else:
        raise ValueError("Unknown kind")

    # Draw transmit power and convert it to amplitude scale.
    tx_power_dbm = draw_tx_power_dbm(kind)
    tx_power_watt = tx_power_dbm_to_watt(tx_power_dbm)
    power_lin = tx_power_watt_to_amplitude_scale(tx_power_watt)

    # Draw initial position in polar coordinates, then convert to X-Y.
    r0 = np.random.uniform(r_min, r_max)
    ang = np.random.uniform(-np.pi, np.pi)

    x0 = r0 * np.cos(ang)
    y0 = r0 * np.sin(ang)
    z0 = np.random.uniform(z_min, z_max)

    pos0 = np.array([x0, y0, z0], dtype=np.float64)

    # Draw velocity magnitude and direction.
    speed = np.random.uniform(speed_min, speed_max)

    if speed == 0.0:
        vel = np.array([0.0, 0.0, 0.0], dtype=np.float64)
    else:
        theta = np.random.uniform(0, 2 * np.pi)
        phi = np.random.uniform(0, np.pi)

        ux = np.sin(phi) * np.cos(theta)
        uy = np.sin(phi) * np.sin(theta)
        uz = np.cos(phi)

        u_vec = np.array([ux, uy, uz], dtype=np.float64)
        vel = speed * u_vec

    f_offset_hz = draw_baseband_frequency_offset()

    return Emitter3D(
        emitter_id=emitter_id,
        kind=kind,
        pos0=pos0,
        vel=vel,
        fc=fc,
        tx_power_dbm=tx_power_dbm,
        tx_power_watt=tx_power_watt,
        power_lin=power_lin,
        f_offset_hz=f_offset_hz,
        sps=sps,
        waveform="QPSK",
    )

def draw_random_counts():
    """
    Draw a random number of objects for one scenario.
    The distributions are chosen to be more realistic:
    - Many pedestrians are possible.
    - Several vehicles are possible.
    - Aircraft are less common, usually 0 to 3.
    """
    num_high = int(np.random.choice(
        [0, 1, 2, 3],
        p=[0.55, 0.30, 0.12, 0.03],
    ))

    num_low = int(min(np.random.poisson(lam=7), 20))
    num_medium = int(min(np.random.poisson(lam=4), 12))
    num_static = int(min(np.random.poisson(lam=2), 6))

    # Make sure that the scenario is not empty.
    if num_static + num_low + num_medium + num_high == 0:
        num_low = 1

    return num_static, num_low, num_medium, num_high

def draw_random_snr():
    """
    Draw a random SNR value for one scenario.
    """
    return float(np.random.uniform(5.0, 25.0))

def create_emitters(num_static, num_low, num_medium, num_high):
    """
    Create all transmitters for one scenario.
    """
    emitters = []
    emitter_id = 0

    for _ in range(num_static):
        emitters.append(random_emitter("static", emitter_id))
        emitter_id += 1

    for _ in range(num_low):
        emitters.append(random_emitter("low", emitter_id))
        emitter_id += 1

    for _ in range(num_medium):
        emitters.append(random_emitter("medium", emitter_id))
        emitter_id += 1

    for _ in range(num_high):
        emitters.append(random_emitter("high", emitter_id))
        emitter_id += 1

    return emitters

# ============================================================
#                    Received signal generation
# ============================================================

def synthesize_emitter_signal_full(
    em: Emitter3D,
    t_abs: np.ndarray,
    rx_pos: np.ndarray = np.array([0.0, 0.0, 0.0]),
) -> np.ndarray:
    """
    Generate the received signal contribution from one transmitter.
    """
    # Position of the emitter as a function of time.
    x_t = em.pos0[0] + em.vel[0] * t_abs
    y_t = em.pos0[1] + em.vel[1] * t_abs
    z_t = em.pos0[2] + em.vel[2] * t_abs

    d_t = np.sqrt(
        (x_t - rx_pos[0]) ** 2 +
        (y_t - rx_pos[1]) ** 2 +
        (z_t - rx_pos[2]) ** 2
    )

    # Doppler is calculated from the radial velocity.
    v_rad_t = radial_velocity_over_time_xyz(
        x_t=x_t,
        y_t=y_t,
        z_t=z_t,
        vel=em.vel,
        rx_pos=rx_pos,
    )

    fd_t = doppler_hz(em.fc, v_rad_t)

    # The instantaneous baseband frequency includes both the selected offset
    # and the Doppler shift.
    f_inst_t = em.f_offset_hz + fd_t

    # Convert frequency to phase using cumulative sum.
    phase_t = 2.0 * np.pi * np.cumsum(f_inst_t, dtype=np.float64) / Fs

    # Generate QPSK signal for this transmitter.
    bb = generate_qpsk_baseband(len(t_abs), sps=em.sps)

    freq_shift = np.exp(1j * phase_t).astype(np.complex64)
    s_t = bb * freq_shift

    # Apply FSPL and transmit power.
    a_fspl = amp_from_fspl(em.fc, d_t).astype(np.float32)
    a_total = a_fspl * np.float32(em.power_lin)

    # Apply statistical fading and multipath.
    shadow_amp, h = generate_channel_fading(
        num_samples=len(t_abs),
        Fs=Fs,
        enable_fading=ENABLE_FADING,
    )

    y_em = a_total * shadow_amp * h * s_t

    return y_em.astype(np.complex64)

def generate_received_signal_full(emitters, snr_db: float | None):
    """
    Sum all transmitter signals and add noise.
    This is the final received signal at the sensor.
    """
    t_abs = np.arange(TOTAL_SAMPLES, dtype=np.float64) / Fs
    y_full = np.zeros(TOTAL_SAMPLES, dtype=np.complex64)

    for em in emitters:
        y_full += synthesize_emitter_signal_full(
            em=em,
            t_abs=t_abs,
        )

    y_full = awgn(y_full, snr_db).astype(np.complex64)

    return y_full

# ============================================================
#                    RSSI and spectrogram
# ============================================================

def compute_rssi_over_time(y: np.ndarray, Fs: float, win_len: int, hop: int):
    """
    Calculate RSSI over time using sliding windows.
    RSSI is calculated here as the mean signal power in each window.
    """
    eps = 1e-18
    starts = np.arange(0, len(y) - win_len + 1, hop)

    rssi_lin = np.empty(len(starts), dtype=np.float64)

    for i, s in enumerate(starts):
        seg = y[s:s + win_len]
        rssi_lin[i] = np.mean(np.abs(seg) ** 2)

    rssi_db = 10.0 * np.log10(rssi_lin + eps)
    t_rssi = (starts + win_len / 2) / Fs

    return t_rssi, rssi_db, rssi_lin

def compute_spectrogram(y: np.ndarray):
    """
    Calculate the STFT spectrogram of the received signal.
    """
    win = get_window(window_name, Nwin, fftbins=True)

    f, tt, Zxx = stft(
        y,
        fs=Fs,
        window=win,
        nperseg=Nwin,
        noverlap=Nwin - hop,
        nfft=Nfft,
        return_onesided=False,
        boundary=None,
        padded=False,
    )

    S_db = 20.0 * np.log10(np.abs(Zxx) + 1e-12)

    return f, tt, Zxx, S_db

# ============================================================
#                    Image saving functions
# ============================================================

def save_spectrogram_image(
    f: np.ndarray,
    tt: np.ndarray,
    S_db: np.ndarray,
    out_dir: Path,
    file_name: str,
    scenario_id: int,
    frame_idx: int,
    frame_start_sec: float,
) -> str:
    """
    Save one spectrogram frame with axes and colorbar.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / file_name

    # Shift negative and positive frequencies to the usual display order.
    f_shift = np.fft.fftshift(f)
    S_shift = np.fft.fftshift(S_db, axes=0)

    # Clip extreme values to make the image easier to read.
    lo, hi = np.percentile(S_shift, [1, 99])
    S_plot = np.clip(S_shift, lo, hi)

    tt_abs = tt + frame_start_sec

    plt.figure(figsize=(9, 5))
    plt.pcolormesh(tt_abs, f_shift / 1e3, S_plot, shading="auto")
    plt.xlabel("Time in scenario [sec]")
    plt.ylabel("Baseband frequency [kHz]")
    plt.title(f"Scenario {scenario_id:06d} - Spectrogram Frame {frame_idx:03d}")
    cbar = plt.colorbar()
    cbar.set_label("Magnitude [dB]")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

    return file_name

def save_scene_map(emitters, out_dir: Path, file_name: str, scenario_id: int):
    """
    Save a 2D top-view map of the scenario.
    This is only for visualization and is not used as the model input.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / file_name

    marker_map = {
        "static": "s",
        "low": "o",
        "medium": "^",
        "high": "*",
    }

    label_map = {
        "static": "Static station",
        "low": "Pedestrian",
        "medium": "Vehicle",
        "high": "Aircraft",
    }

    size_map = {
        "static": 70,
        "low": 35,
        "medium": 55,
        "high": 130,
    }

    plt.figure(figsize=(8, 8))
    ax = plt.gca()

    ax.scatter(0, 0, marker="X", s=180, label="Sensor / Receiver")

    for kind in ["static", "low", "medium", "high"]:
        group = [e for e in emitters if e.kind == kind]

        if len(group) == 0:
            continue

        xs = np.array([e.pos0[0] for e in group])
        ys = np.array([e.pos0[1] for e in group])

        ax.scatter(
            xs,
            ys,
            marker=marker_map[kind],
            s=size_map[kind],
            label=f"{label_map[kind]} ({len(group)})",
            alpha=0.85,
        )

        # Draw the movement direction during the scenario.
        for e in group:
            start_xy = e.pos0[:2]
            end_pos = e.pos0 + e.vel * SCENARIO_DURATION_SEC
            end_xy = end_pos[:2]

            dx = end_xy[0] - start_xy[0]
            dy = end_xy[1] - start_xy[1]

            if np.sqrt(dx**2 + dy**2) > 1e-6:
                ax.arrow(
                    start_xy[0],
                    start_xy[1],
                    dx,
                    dy,
                    head_width=35,
                    head_length=45,
                    length_includes_head=True,
                    alpha=0.4,
                )

    # Set plot limits according to all object positions.
    if len(emitters) > 0:
        all_x_start = np.array([e.pos0[0] for e in emitters])
        all_y_start = np.array([e.pos0[1] for e in emitters])

        all_end = np.array([e.pos0 + e.vel * SCENARIO_DURATION_SEC for e in emitters])
        all_x_end = all_end[:, 0]
        all_y_end = all_end[:, 1]

        all_x = np.concatenate([all_x_start, all_x_end])
        all_y = np.concatenate([all_y_start, all_y_end])

        max_range = max(np.max(np.abs(all_x)), np.max(np.abs(all_y)), 500.0)
    else:
        max_range = 500.0

    max_range *= 1.15

    ax.set_xlim(-max_range, max_range)
    ax.set_ylim(-max_range, max_range)

    ax.set_xlabel("X position [m]")
    ax.set_ylabel("Y position [m]")
    ax.set_title(f"2D Environment Map - Scenario {scenario_id:06d}")
    ax.grid(True)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="upper right", fontsize=8)

    ax.text(
        0.02,
        0.02,
        "Top view (XY plane)\nArrows show movement during scenario",
        transform=ax.transAxes,
        fontsize=9,
        bbox=dict(facecolor="white", alpha=0.75),
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

    return file_name

# ============================================================
#                    Metadata helpers
# ============================================================

def safe_mean(values):
    return float(np.mean(values)) if len(values) > 0 else ""

def safe_min(values):
    return float(np.min(values)) if len(values) > 0 else ""

def safe_max(values):
    return float(np.max(values)) if len(values) > 0 else ""

def build_kind_summary(emitters):
    """
    Build a short summary for every object type.
    This makes the metadata easier to analyze later.
    """
    summary = {}

    for kind in ["static", "low", "medium", "high"]:
        group = [e for e in emitters if e.kind == kind]

        speeds = [float(np.linalg.norm(e.vel)) for e in group]
        tx_dbm = [e.tx_power_dbm for e in group]
        tx_watt = [e.tx_power_watt for e in group]
        power_lin = [e.power_lin for e in group]
        offsets = [e.f_offset_hz for e in group]

        summary[kind] = {
            "count": len(group),
            "sps": SPS_BY_KIND[kind],
            "symbol_rate_hz_approx": Fs / SPS_BY_KIND[kind],
            "tx_power_dbm_min": safe_min(tx_dbm),
            "tx_power_dbm_mean": safe_mean(tx_dbm),
            "tx_power_dbm_max": safe_max(tx_dbm),
            "tx_power_watt_mean": safe_mean(tx_watt),
            "power_lin_mean": safe_mean(power_lin),
            "speed_mps_mean": safe_mean(speeds),
            "baseband_offset_hz_mean": safe_mean(offsets),
        }

    return summary

def build_metadata_row(
    scenario_id: int,
    frame_files,
    scene_file: str,
    emitters,
    rssi_file: str,
    rssi_stats: dict,
    snr_db: float | None,
    seed: int,
):
    """
    Create one metadata row for a full scenario.
    """
    labels = {
        "static": sum(1 for e in emitters if e.kind == "static"),
        "low": sum(1 for e in emitters if e.kind == "low"),
        "medium": sum(1 for e in emitters if e.kind == "medium"),
        "high": sum(1 for e in emitters if e.kind == "high"),
    }

    emitters_list = []

    for e in emitters:
        speed = float(np.linalg.norm(e.vel))

        pos_start = e.pos0
        pos_end = e.pos0 + e.vel * SCENARIO_DURATION_SEC

        v_rad_start = radial_velocity_single(pos_start, e.vel)
        v_rad_end = radial_velocity_single(pos_end, e.vel)

        fd_start = float(doppler_hz(e.fc, v_rad_start))
        fd_end = float(doppler_hz(e.fc, v_rad_end))

        emitters_list.append({
            "emitter_id": e.emitter_id,
            "kind": e.kind,
            "pos0_m": e.pos0.tolist(),
            "pos_end_m": pos_end.tolist(),
            "vel_mps": e.vel.tolist(),
            "speed_mps": speed,
            "fc_hz": e.fc,
            "tx_power_dbm": float(e.tx_power_dbm),
            "tx_power_watt": float(e.tx_power_watt),
            "power_lin": float(e.power_lin),
            "sps": int(e.sps),
            "symbol_rate_hz_approx": float(Fs / e.sps),
            "baseband_offset_hz": float(e.f_offset_hz),
            "baseband_offset_khz": float(e.f_offset_hz / 1e3),
            "radial_velocity_start_mps": v_rad_start,
            "radial_velocity_end_mps": v_rad_end,
            "doppler_start_hz": fd_start,
            "doppler_end_hz": fd_end,
        })

    kind_summary = build_kind_summary(emitters)

    channel_params = {
        "enable_fading": ENABLE_FADING,
        "shadow_std_db": SHADOW_STD_DB,
        "shadow_update_sec": SHADOW_UPDATE_SEC,
        "fading_update_sec": FADING_UPDATE_SEC,
        "rician_k_db": DEFAULT_K_DB,
        "continuous_full_signal": True,
        "sps_by_kind": SPS_BY_KIND,
        "tx_power_dbm_range_by_kind": TX_POWER_DBM_RANGE_BY_KIND,
    }

    row = {
        "scenario_id": scenario_id,

        "labels": json.dumps(labels),
        "num_static": labels["static"],
        "num_low": labels["low"],
        "num_medium": labels["medium"],
        "num_high": labels["high"],
        "num_emitters": len(emitters),

        "sps_static": SPS_BY_KIND["static"],
        "sps_low": SPS_BY_KIND["low"],
        "sps_medium": SPS_BY_KIND["medium"],
        "sps_high": SPS_BY_KIND["high"],

        "symbol_rate_static_hz": Fs / SPS_BY_KIND["static"],
        "symbol_rate_low_hz": Fs / SPS_BY_KIND["low"],
        "symbol_rate_medium_hz": Fs / SPS_BY_KIND["medium"],
        "symbol_rate_high_hz": Fs / SPS_BY_KIND["high"],

        "tx_power_static_dbm_min": kind_summary["static"]["tx_power_dbm_min"],
        "tx_power_static_dbm_mean": kind_summary["static"]["tx_power_dbm_mean"],
        "tx_power_static_dbm_max": kind_summary["static"]["tx_power_dbm_max"],

        "tx_power_low_dbm_min": kind_summary["low"]["tx_power_dbm_min"],
        "tx_power_low_dbm_mean": kind_summary["low"]["tx_power_dbm_mean"],
        "tx_power_low_dbm_max": kind_summary["low"]["tx_power_dbm_max"],

        "tx_power_medium_dbm_min": kind_summary["medium"]["tx_power_dbm_min"],
        "tx_power_medium_dbm_mean": kind_summary["medium"]["tx_power_dbm_mean"],
        "tx_power_medium_dbm_max": kind_summary["medium"]["tx_power_dbm_max"],

        "tx_power_high_dbm_min": kind_summary["high"]["tx_power_dbm_min"],
        "tx_power_high_dbm_mean": kind_summary["high"]["tx_power_dbm_mean"],
        "tx_power_high_dbm_max": kind_summary["high"]["tx_power_dbm_max"],

        "power_lin_static_mean": kind_summary["static"]["power_lin_mean"],
        "power_lin_low_mean": kind_summary["low"]["power_lin_mean"],
        "power_lin_medium_mean": kind_summary["medium"]["power_lin_mean"],
        "power_lin_high_mean": kind_summary["high"]["power_lin_mean"],

        "speed_static_mps_mean": kind_summary["static"]["speed_mps_mean"],
        "speed_low_mps_mean": kind_summary["low"]["speed_mps_mean"],
        "speed_medium_mps_mean": kind_summary["medium"]["speed_mps_mean"],
        "speed_high_mps_mean": kind_summary["high"]["speed_mps_mean"],

        "spectrogram_sequence_files": json.dumps(frame_files),
        "scene_file": scene_file,
        "rssi_file": rssi_file,
        "rssi_stats": json.dumps(rssi_stats),

        "kind_summary": json.dumps(kind_summary),
        "emitters": json.dumps(emitters_list),

        "Fs": Fs,
        "scenario_duration_sec": SCENARIO_DURATION_SEC,
        "frame_duration_sec": FRAME_DURATION_SEC,
        "frame_step_sec": FRAME_STEP_SEC,
        "num_frames": NUM_FRAMES,
        "total_samples": TOTAL_SAMPLES,
        "frame_samples": FRAME_SAMPLES,
        "Nwin": Nwin,
        "hop": hop,
        "Nfft": Nfft,
        "window": window_name,

        "snr_db": snr_db,
        "seed": seed,
        "channel_params": json.dumps(channel_params),
    }

    return row

def append_metadata_row(csv_path: Path, row: dict):
    """
    Append one scenario row to the metadata CSV file.
    """
    fieldnames = [
        "scenario_id",

        "labels",
        "num_static",
        "num_low",
        "num_medium",
        "num_high",
        "num_emitters",

        "sps_static",
        "sps_low",
        "sps_medium",
        "sps_high",

        "symbol_rate_static_hz",
        "symbol_rate_low_hz",
        "symbol_rate_medium_hz",
        "symbol_rate_high_hz",

        "tx_power_static_dbm_min",
        "tx_power_static_dbm_mean",
        "tx_power_static_dbm_max",

        "tx_power_low_dbm_min",
        "tx_power_low_dbm_mean",
        "tx_power_low_dbm_max",

        "tx_power_medium_dbm_min",
        "tx_power_medium_dbm_mean",
        "tx_power_medium_dbm_max",

        "tx_power_high_dbm_min",
        "tx_power_high_dbm_mean",
        "tx_power_high_dbm_max",

        "power_lin_static_mean",
        "power_lin_low_mean",
        "power_lin_medium_mean",
        "power_lin_high_mean",

        "speed_static_mps_mean",
        "speed_low_mps_mean",
        "speed_medium_mps_mean",
        "speed_high_mps_mean",

        "spectrogram_sequence_files",
        "scene_file",
        "rssi_file",
        "rssi_stats",

        "kind_summary",
        "emitters",

        "Fs",
        "scenario_duration_sec",
        "frame_duration_sec",
        "frame_step_sec",
        "num_frames",
        "total_samples",
        "frame_samples",
        "Nwin",
        "hop",
        "Nfft",
        "window",

        "snr_db",
        "seed",
        "channel_params",
    ]

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    need_header = not csv_path.exists()

    with csv_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)

        if need_header:
            writer.writeheader()

        writer.writerow(row)
        f.flush()

# ============================================================
#                    One scenario generation
# ============================================================

def generate_one_scenario(scenario_id: int, base_seed: int = 12345):
    """
    Generate and save one complete scenario.
    """
    seed = base_seed + scenario_id
    np.random.seed(seed)

    num_static, num_low, num_medium, num_high = draw_random_counts()
    snr_db = draw_random_snr()

    emitters = create_emitters(
        num_static=num_static,
        num_low=num_low,
        num_medium=num_medium,
        num_high=num_high,
    )

    # Generate one continuous signal for the full scenario.
    y_full = generate_received_signal_full(
        emitters=emitters,
        snr_db=snr_db,
    )

    # Calculate RSSI for the full scenario.
    t_rssi, rssi_db, rssi_lin = compute_rssi_over_time(
        y=y_full,
        Fs=Fs,
        win_len=Nwin,
        hop=hop,
    )

    rssi_stats = {
        "mean_db": float(np.mean(rssi_db)),
        "std_db": float(np.std(rssi_db)),
        "min_db": float(np.min(rssi_db)),
        "max_db": float(np.max(rssi_db)),
    }

    rssi_file = f"scenario_{scenario_id:06d}_rssi.json"
    rssi_path = OUT_RSSI_DIR / rssi_file

    with rssi_path.open("w", encoding="utf-8") as f_json:
        json.dump(
            {
                "t_sec": t_rssi.tolist(),
                "rssi_db": rssi_db.tolist(),
                "rssi_lin": rssi_lin.tolist(),
            },
            f_json,
            ensure_ascii=False,
        )

    # Save the spectrogram sequence.
    scenario_spec_dir = OUT_SPEC_DIR / f"scenario_{scenario_id:06d}"
    scenario_spec_dir.mkdir(parents=True, exist_ok=True)

    frame_files = []

    for frame_idx in range(NUM_FRAMES):
        start_sample = frame_idx * FRAME_STEP_SAMPLES
        end_sample = start_sample + FRAME_SAMPLES

        frame_start_sec = start_sample / Fs
        y_frame = y_full[start_sample:end_sample]

        f, tt, Zxx, S_db = compute_spectrogram(y_frame)

        frame_file_name = f"frame_{frame_idx:03d}.png"

        save_spectrogram_image(
            f=f,
            tt=tt,
            S_db=S_db,
            out_dir=scenario_spec_dir,
            file_name=frame_file_name,
            scenario_id=scenario_id,
            frame_idx=frame_idx,
            frame_start_sec=frame_start_sec,
        )

        relative_frame_path = (
            f"spectrogram_sequences/scenario_{scenario_id:06d}/{frame_file_name}"
        )
        frame_files.append(relative_frame_path)

    # Save a 2D map of the simulated environment.
    scene_file = f"scenario_{scenario_id:06d}_scene.png"

    save_scene_map(
        emitters=emitters,
        out_dir=OUT_SCENE_DIR,
        file_name=scene_file,
        scenario_id=scenario_id,
    )

    scene_relative_path = f"scene_maps/{scene_file}"
    rssi_relative_path = f"rssi/{rssi_file}"

    row = build_metadata_row(
        scenario_id=scenario_id,
        frame_files=frame_files,
        scene_file=scene_relative_path,
        emitters=emitters,
        rssi_file=rssi_relative_path,
        rssi_stats=rssi_stats,
        snr_db=snr_db,
        seed=seed,
    )

    append_metadata_row(OUT_META_CSV, row)

    print(
        f"scenario {scenario_id:06d} saved | "
        f"frames={NUM_FRAMES} | "
        f"labels={{static:{num_static}, low:{num_low}, medium:{num_medium}, high:{num_high}}} | "
        f"SNR={snr_db:.2f} dB | "
        f"RSSI mean={rssi_stats['mean_db']:.2f} dB"
    )

# ============================================================
#                    Dataset generation
# ============================================================

def get_next_scenario_id():
    """
    Find the next available scenario ID.

    I added this function because I want to run the simulator many times
    in small batches, without deleting the old samples and without
    overwriting previous scenarios.
    """
    max_id = -1

    # First, check the existing spectrogram scenario folders.
    if OUT_SPEC_DIR.exists():
        for folder in OUT_SPEC_DIR.glob("scenario_*"):
            if folder.is_dir():
                try:
                    scenario_id = int(folder.name.replace("scenario_", ""))
                    max_id = max(max_id, scenario_id)
                except ValueError:
                    # If the folder name is not in the correct format, ignore it.
                    pass

    # Also check the metadata file, because it contains all saved scenarios.
    if OUT_META_CSV.exists():
        with OUT_META_CSV.open("r", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            for row in reader:
                try:
                    scenario_id = int(row["scenario_id"])
                    max_id = max(max_id, scenario_id)
                except Exception:
                    # If there is a problem with one row, continue checking the rest.
                    pass

    return max_id + 1

def reset_output_dirs():
    """
    Delete the previous dataset folder and create a clean one.
    """
    if OUT_BASE_DIR.exists():
        shutil.rmtree(OUT_BASE_DIR)

    OUT_SPEC_DIR.mkdir(parents=True, exist_ok=True)
    OUT_RSSI_DIR.mkdir(parents=True, exist_ok=True)
    OUT_SCENE_DIR.mkdir(parents=True, exist_ok=True)

def generate_dataset(
    num_scenarios: int = 10,
    reset_dirs: bool = False,
    base_seed: int = 12345,
):
    """
    Generate a batch of scenarios.

    If reset_dirs=True:
        The old dataset is deleted and the scenario IDs start again from 0.

    If reset_dirs=False:
        The code continues from the next available scenario ID.
    """
    if reset_dirs:
        # When I want to start a new clean dataset.
        reset_output_dirs()
        start_id = 0
    else:
        # Do not delete old data. Just make sure the folders exist.
        OUT_SPEC_DIR.mkdir(parents=True, exist_ok=True)
        OUT_RSSI_DIR.mkdir(parents=True, exist_ok=True)
        OUT_SCENE_DIR.mkdir(parents=True, exist_ok=True)

        # Continue from the next scenario number.
        start_id = get_next_scenario_id()

    print(f"Starting from scenario id: {start_id}")
    print(f"Generating {num_scenarios} new scenarios...")

    for i in range(num_scenarios):
        scenario_id = start_id + i

        generate_one_scenario(
            scenario_id=scenario_id,
            base_seed=base_seed,
        )

    print("Batch generation completed.")
    print("Base directory:", OUT_BASE_DIR)
    print("Spectrogram sequences:", OUT_SPEC_DIR)
    print("Scene maps:", OUT_SCENE_DIR)
    print("RSSI:", OUT_RSSI_DIR)
    print("Metadata:", OUT_META_CSV)

# ============================================================
#                             Main
# ============================================================

def main():

    generate_dataset(
        num_scenarios=10,
        reset_dirs=False,      # keep old samples and add new ones
        base_seed=12345,
    )


if __name__ == "__main__":
    main()


Starting from scenario id: 500
Generating 10 new scenarios...
scenario 000500 saved | frames=8 | labels={static:2, low:5, medium:2, high:1} | SNR=10.15 dB | RSSI mean=-74.87 dB
scenario 000501 saved | frames=8 | labels={static:4, low:4, medium:5, high:0} | SNR=15.33 dB | RSSI mean=-61.12 dB
scenario 000502 saved | frames=8 | labels={static:1, low:6, medium:2, high:0} | SNR=15.83 dB | RSSI mean=-64.87 dB
scenario 000503 saved | frames=8 | labels={static:3, low:2, medium:5, high:2} | SNR=15.33 dB | RSSI mean=-62.78 dB
scenario 000504 saved | frames=8 | labels={static:4, low:6, medium:5, high:2} | SNR=15.56 dB | RSSI mean=-60.37 dB
scenario 000505 saved | frames=8 | labels={static:1, low:13, medium:3, high:0} | SNR=11.70 dB | RSSI mean=-69.76 dB
scenario 000506 saved | frames=8 | labels={static:1, low:7, medium:7, high:0} | SNR=12.56 dB | RSSI mean=-58.13 dB
scenario 000507 saved | frames=8 | labels={static:3, low:10, medium:3, high:0} | SNR=24.38 dB | RSSI mean=-62.11 dB
scenario 000508 

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/my_project/data")

print("Base exists:", BASE_DIR.exists())
print("Metadata exists:", (BASE_DIR / "metadata.csv").exists())
print("Metadata path:", BASE_DIR / "metadata.csv")

print("Spectrograms:", len(list((BASE_DIR / "spectrogram_sequences").glob("*.png"))))
print("Scene maps:", len(list((BASE_DIR / "scene_maps").glob("*.png"))))
print("RSSI files:", len(list((BASE_DIR / "rssi").glob("*.json"))))

Base exists: True
Metadata exists: True
Metadata path: /content/drive/MyDrive/my_project/data/metadata.csv
Spectrograms: 0
Scene maps: 509
RSSI files: 509
